In [1]:
from langchain_community.document_loaders import WebBaseLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter, HTMLHeaderTextSplitter, Language
from langchain.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFaceEndpoint, ChatHuggingFace
from bs4 import BeautifulSoup
from dotenv import load_dotenv
import os

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [2]:
load_dotenv()

True

In [3]:
url = 'https://www.flipkart.com/motorola-motobook-60-full-metal-oled-i5-14th-gen-intel-core-5-series-2-210h-16-gb-512-gb-ssd-windows-11-home-14irh10r-thin-light-laptop/p/itm9a50f9400e0e0?pid=COMHAUZWVNJSFAMN&otracker=wishlist&lid=LSTCOMHAUZWVNJSFAMNCQMECQ&fm=organic&iid=eb8ce955-00a1-4fdc-bd17-e3cc38eb95b2.COMHAUZWVNJSFAMN.PRODUCTSUMMARY&ppt=hp&ppn=homepage&ssid=txin2y7f340000001758805452784'

In [4]:
loader = WebBaseLoader(url)
raw_html = loader.scrape()

In [ ]:
for tag in raw_html([
    "script", "style", "nav", "footer", "header", "aside", "meta", "link",
    "noscript",   # fallback content, usually redundant
    "iframe",     # embedded ads, videos
    "form",       # login/signup/contact forms
    "input", "button", "select", "textarea",  # form fields
    "svg", "canvas",  # icons, graphics
    "img",       # images (unless you want alt text)
    "video", "audio", "source", "track",  # media elements
    "advertisement", "ads",  # ad containers (if present as tags)
]):
    tag.decompose()

raw_html = str(raw_html)

In [7]:
soup = BeautifulSoup(raw_html, "html.parser")

# remove all attributes from all tags
for tag in soup.find_all(True):  # True = all tags
    tag.attrs = {}

raw_html = str(soup)

In [8]:
# Split each section by HTML-aware splitter
html_splitter = RecursiveCharacterTextSplitter.from_language(
    language=Language.HTML,
    chunk_size=5000,
    chunk_overlap=500
)

html_chunks = html_splitter.split_text(raw_html)

print(f"Number of HTML-aware chunks: {len(html_chunks)}")

Number of HTML-aware chunks: 8


In [9]:
html_chunks[4]

"<div>Design</div></div></a><a><div><div></div><div>Display</div></div></a></div></div></div><div><div><div><div></div><div></div><div></div><div></div><div></div><div></div><div></div><div><span>+ <!-- -->52</span></div></div></div></div><div><div><div><div><div><div>5</div><p>Brilliant</p></div><div><div><div><div>Great product moto killed it peak performance student must have this product I am the first review in flipkart</div><span><span>READ MORE</span></span></div></div></div><div><div></div><div></div></div><div><div><p>Mohamed  Sadiq </p><p><span>Certified Buyer</span><span>, Ramanathapuram</span></p><div></div><p>4 months ago</p></div><div><div><div><div><span>133</span></div><div><span>25</span></div></div><div><div><div><a><span>Permalink</span></a></div><div><span>Report Abuse</span></div></div></div></div></div></div></div></div></div><div><div><div><div><div>4</div><p>Delightful</p></div><div><div><div><div>It is a good laptop for moderate browsing and light gaming users.

In [10]:
clean_chunks = []
for text in html_chunks:
    soup = BeautifulSoup(text, "html.parser")
    clean_text = soup.get_text(separator="<<>>", strip=True)
    if len(clean_text) > 5:
        clean_chunks.append(clean_text)

lengths = []
for text in clean_chunks:
    lengths.append(len(text))

print(sorted(lengths))

# print(html_chunks[45])
# print('\n\n')
print(clean_chunks[-2])


[181, 1823, 1941, 2016, 2682, 2936, 2984]
+<<>>All 76 reviews<<>>Questions and Answers<<>>Q:<<>>Does it have WiFi 7..??<<>>A:<<>>Yes motobook 60 have wifi 7 and speed is quite    impressive<<>>Rahul  Khatkar<<>>Certified Buyer<<>>3<<>>0<<>>Report Abuse<<>>Q:<<>>Does it have HDR Certification?<<>>A:<<>>Yes<<>>Rahul  Khatkar<<>>Certified Buyer<<>>2<<>>1<<>>Report Abuse<<>>Q:<<>>What are the contents in box<<>>A:<<>>Laptop, charger and user manual<<>>Rahul  Khatkar<<>>Certified Buyer<<>>1<<>>0<<>>Report Abuse<<>>Q:<<>>How is the Battery life......<<>>A:<<>>For normal if you are using YouTube or powerpoint or surfing it is around 6 hrs approx. Heavy task like editing and all 2 hrs to 3hrs.<<>>Anonymous<<>>Certified Buyer<<>>33<<>>8<<>>Report Abuse<<>>Q:<<>>Does it have lifetime ms office<<>>A:<<>>Yes<<>>Demon King<<>>Certified Buyer<<>>1<<>>0<<>>Report Abuse<<>>Read other answers<<>>Q:<<>>I saw many negative reviews on sound quality, how it is?<<>>A:<<>>Speakers aren't that loud but they a

In [ ]:
final_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["<<>>", "\n\n\n", "\n\n", "\n", ".", " ", ""]
)
final_chunks = []
for text in clean_chunks:
    new_chunks = final_splitter.split_text(text)
    final_chunks.extend(new_chunks)

len(final_chunks)

chunks = []
for text in final_chunks:
    new_text = text.replace("<<>>", " ")
    chunks.append(new_text)

# for text in chunks:
#     print(text)

Explore Plus Login Become a Seller More Cart Home Computers Laptops MOTOROLA Laptops MOTOROLA Motobook 60 Full Metal OLED (i5 14th Gen) Intel Core 5 (Series 2) 210H - (16 GB/512 GB SSD/Windows 11 Home) 14IRH10R Thin and Light Laptop (14 Inch, PANTONE Wedgewood, 1.4 Kg, With MS Office) Compare Share MOTOROLA Motobook 60 Full Metal OLED (i5 14th Gen) Intel Core 5 (Series 2) 210H - (16 GB/512 GB SSD/Windows 11 Home) 14IRH10R Thin and Light Laptop (14 Inch, PANTONE Wedgewood, 1.4 Kg, With MS Office) 4.4 715 Ratings & 76 Reviews Special price ₹49,989 ₹ 93,690 46% off + ₹99 Protect Promise Fee Learn more Secure delivery by 4 Oct, Saturday Available offers Bank Offer 10% Off on Supermoney UPI. Max discount of ₹50. Minimum order value of ₹250. T&C Bank Offer 5% cashback on Flipkart SBI Credit Card upto ₹4,000 per calendar quarter T&C Bank Offer 5% cashback on Axis Bank Flipkart Debit Card
 T&C Bank Offer 5% cashback on Flipkart SBI Credit Card upto ₹4,000 per calendar quarter T&C Bank Offer 5%

In [12]:
embeddings = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')
vectorstore = FAISS.from_texts(chunks, embeddings)

d:\anaconda3\envs\genai_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [40]:
retriever = vectorstore.as_retriever(search_type='similarity', search_kwargs={'k':5})

In [16]:
from langchain.memory import VectorStoreRetrieverMemory
memory_store = FAISS.from_texts([""], embeddings)
memory = VectorStoreRetrieverMemory(retriever=memory_store.as_retriever())

C:\Users\pc\AppData\Local\Temp\ipykernel_4900\3054852336.py:3: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = VectorStoreRetrieverMemory(retriever=memory_store.as_retriever())


In [42]:
ans = retriever.invoke('what are the reviews ?')
ans

[Document(id='8e99219b-f562-4343-9d59-633f9593bd9b', metadata={}, page_content=' 4.4 ★ 715 Ratings & 76 Reviews 5 ★ 4 ★ 3 ★ 2 ★ 1 ★ 448 177 35 7 48 Performance Battery Design Display + 52 5 Brilliant Great product moto killed it peak performance student must have this product I am the first review in flipkart READ MORE'),
 Document(id='f4f67aa4-6526-4356-b576-eaab69d06815', metadata={}, page_content=' Report Abuse 5 Excellent Good choice under 55k. Pros: Oled Screen is awesome. It supports hdr. Fantastic for entertainment purposes. Performance is good, no issues there. Battery backup is satisfactory. If you want more backup, try energy saving mode and reduce display resolution to 1080p.(6+hrs) Ram and storage are expandable. Cons: Sometimes it heats up but normally it stays cool. Try not to use it on lap ot any other surface where air flow is difficult. Battery drains fast without energy saving mode.(3-4hrs) ... READ MORE Junaid Certified Buyer , Ajmer 3 months ago 12 2 Permalink Repor

In [43]:
prompt_template = """
You are a helpful assistant.

Conversation history (retrieved from memory):
{history}

Relevant Webpage Content context:
{context}

User Query: {query}

Don't explain what are you doing to answer the query or how are you doing things just give the answer the user needs.
Answer clearly using both conversation history and webpage content context.
If the user asks for a summary/overview, summarize the whole webpage content.
If the context doesn't contain the relevant information to answer the user query, then say something like Webpage doesn't contain the relevant information.
"""

In [44]:
prompt = PromptTemplate(
    template=prompt_template,
    input_variables=["history", "context", "query"]
)

prompt

PromptTemplate(input_variables=['context', 'history', 'query'], input_types={}, partial_variables={}, template="\nYou are a helpful assistant.\n\nConversation history (retrieved from memory):\n{history}\n\nRelevant Webpage Content context:\n{context}\n\nUser Query: {query}\n\nDon't explain what are you doing to answer the query or how are you doing things just give the answer the user needs.\nAnswer clearly using both conversation history and webpage content context.\nIf the user asks for a summary/overview, summarize the whole webpage content.\nIf the context doesn't contain the relevant information to answer the user query, then say something like Webpage doesn't contain the relevant information.\n")

In [45]:
llm = HuggingFaceEndpoint(
    model="meta-llama/Llama-3.3-70B-Instruct",
    task="text-generation",
    temperature=0
)

model = ChatHuggingFace(llm=llm)

In [46]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [ ]:
def format_docs(retrieved_docs):
    return "\n\n".join(
        f"[Chunk {i}] {doc.page_content}" for i, doc in enumerate(retrieved_docs, 1)
    )

In [48]:
parser = StrOutputParser()

In [49]:
def get_history(query: str):
    history_docs = memory.load_memory_variables({'input':query})
    return history_docs.get("history", "")

In [50]:
parallel_chain = RunnableParallel({
    'history': RunnableLambda(get_history),
    'context': retriever | RunnableLambda(format_docs),
    'query': RunnablePassthrough()
})

In [51]:
main_chain = parallel_chain | prompt | model | parser

In [52]:
def ask(query: str):
    answer = main_chain.invoke(query)
    memory.save_context({"input": query}, {"output": answer})
    return answer

In [53]:
ask("What is the price of this product?")

'₹49,989'

In [54]:
ask("what was the first question i asked")

'The first question you asked was "What is the price of this product?" and the answer was ₹49,989.'

In [55]:
ask("can you provide me some user revviews backing the first question")

'Based on your first question about the price of the product, which is ₹49,989, here are some user reviews: \n448 people rated it 5★, 177 people rated it 4★, 35 people rated it 3★, 7 people rated it 2★, and 48 people rated it 1★. \nOne review says "Brilliant Great product moto killed it peak performance student must have this product".'

In [56]:
ask("no i was asking the reviews that mentions about its price ")

'One review says "I got in 53k, with emi discount final price was 47k only, it\'s a deal, with this price go for it without second thought". Another review does not specifically mention the price but mentions it is "Worth every penny".'

In [57]:
ask("so aren't there any else ?")

'There are other reviews that mention the price, such as one review saying "Worth every penny" and another review mentioning the price as ₹47,000 after EMI discount. Additionally, the product is currently priced at ₹49,989 with a 46% discount from the original price of ₹93,690. \n\nOther user reviews include: \nOne review says "Brilliant Great product moto killed it peak performance student must have this product".\nAnother review mentions "Cool things: Power button is on the side Could be better: Keyboard layout: page up/down, home,end  dedicated keys, fingerprint sensor Speakers Battery backup is (5+hrs)".\nA review from Simant Certified Buyer says "Decent product" and mentions some suggestions for Moto to increase sales. \n\nThere are a total of 715 ratings and 76 reviews for this product, with 448 people rating it 5-star and 177 people rating it 4-star.'